# 📡 TP Télécom 3GPP — Phase 6 : RAG + Agent

**Objectif** : Transformer le pipeline RAG en un Agent capable de :
- **Décider** quand chercher dans la base de connaissances
- **Planifier** sa réponse en plusieurs étapes
- **Utiliser des outils** : recherche, calcul, vérification

**Architecture Agent :**
```
Question → Agent → [Décision]
                      ↓
              ┌── Outil 1 : Recherche RAG
              ├── Outil 2 : Vérification catégorie
              └── Outil 3 : Génération réponse
                      ↓
                  Réponse finale
```

```
Phase 1 ✅ → Phase 2 ✅ → Phase 3 ✅ → Phase 4 ✅ → Phase 5 ✅ → [Phase 6] → Phase 7
```

## 1. Installation des dépendances

In [ ]:
!pip install -q transformers torch
!pip install -q faiss-cpu sentence-transformers rank-bm25
!pip install -q evaluate rouge_score
!pip install -q pandas matplotlib
print('✅ Installation terminée')

## 2. Imports & Configuration

In [ ]:
import json, time
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = r'C:\Users\HP\Documents\TP-LLM-3GPP-Pipeline'
print('✅ Imports effectués')

## 3. Chargement des Ressources

In [ ]:
# Config
with open(f'{SAVE_DIR}\\pipeline_config.json') as f:
    config = json.load(f)
MODEL_ID = config['best_model_id']
print(f'✅ Modèle : {config["best_model_name"]}')

# LLM
print('🔄 Chargement LLM...')
llm = pipeline('text-generation', model=MODEL_ID, pad_token_id=50256)
print('✅ LLM chargé')

# Embedding
print('🔄 Chargement embedding...')
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('✅ Embedding chargé')

# Dataset
TRAIN_PATH = r'C:\Users\HP\Downloads\TeleQnA_training.txt'
TEST_PATH  = r'C:\Users\HP\Downloads\TeleQnA_testing.txt'
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)
with open(TEST_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)
print(f'✅ Dataset chargé')

## 4. Construction de la Base de Connaissances

In [ ]:
# Corpus
corpus = []
for key, item in train_data.items():
    answer_idx  = item.get('answer', 1)
    answer_text = item.get(f'option {answer_idx}', str(answer_idx))
    corpus.append({
        'text'    : f"Question: {item['question']} Answer: {answer_text}. {item.get('explanation', '')}",
        'category': item.get('category', '3GPP'),
        'answer'  : str(answer_text)
    })

# Index FAISS
texts      = [doc['text'] for doc in corpus]
embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Index BM25
tokenized = [doc['text'].lower().split() for doc in corpus]
bm25      = BM25Okapi(tokenized)

print(f'✅ Base de connaissances : {len(corpus)} documents')
print(f'✅ Index FAISS + BM25 créés')

## 5. Définition des Outils de l'Agent

In [ ]:
# ============================================================
# OUTIL 1 : Recherche RAG (Dense + Sparse)
# ============================================================
def tool_search(query, top_k=3):
    """Recherche les documents les plus pertinents."""
    # Dense (FAISS)
    q_emb = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, top_k)
    results = [{
        'text'    : corpus[i]['text'],
        'category': corpus[i]['category'],
        'score'   : float(s)
    } for s, i in zip(scores[0], indices[0])]
    return results

# ============================================================
# OUTIL 2 : Détection de catégorie
# ============================================================
def tool_detect_category(question):
    """Détecte la catégorie 3GPP de la question."""
    keywords = {
        'RRC/Radio'    : ['rrc', 'radio', 'phy', 'mac', 'pdcp', 'rlc', 'nr', 'lte'],
        'Core Network' : ['amf', 'smf', 'upf', 'pcf', 'udm', 'core', '5gc', 'epc'],
        'Security'     : ['security', 'authentication', 'encryption', 'integrity', 'aka'],
        'Services'     : ['service', 'slice', 'qos', 'bearer', 'session', 'pdu'],
        'General 3GPP' : ['3gpp', 'release', 'standard', 'specification']
    }
    q_lower = question.lower()
    scores  = {}
    for cat, words in keywords.items():
        scores[cat] = sum(1 for w in words if w in q_lower)
    return max(scores, key=scores.get)

# ============================================================
# OUTIL 3 : Génération de réponse
# ============================================================
def tool_generate(question, context, max_new_tokens=100):
    """Génère une réponse avec le contexte fourni."""
    prompt = f'3GPP Context: {context[:300]} Question: {question} Answer:'
    result = llm(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=50256,
        truncation=True
    )
    answer = result[0]['generated_text'].replace(prompt, '').strip()
    return answer

# ============================================================
# OUTIL 4 : Vérification de confiance
# ============================================================
def tool_confidence(question, docs):
    """Calcule un score de confiance basé sur la pertinence des docs."""
    if not docs:
        return 0.0
    avg_score = sum(d['score'] for d in docs) / len(docs)
    return round(float(avg_score), 4)

print('✅ 4 outils définis :')
print('   - tool_search      : Recherche RAG')
print('   - tool_detect_category : Détection catégorie')
print('   - tool_generate    : Génération réponse')
print('   - tool_confidence  : Score de confiance')

## 6. Agent RAG — Pipeline Complet

In [ ]:
def rag_agent(question, verbose=True):
    """
    Agent RAG complet avec raisonnement en plusieurs étapes :
    1. Détection catégorie
    2. Recherche RAG
    3. Évaluation confiance
    4. Génération réponse
    """
    t0 = time.time()
    trace = []  # Journal des actions de l'agent

    # ── Étape 1 : Détection de catégorie ──
    category = tool_detect_category(question)
    trace.append(f'[Étape 1] Catégorie détectée : {category}')
    if verbose:
        print(f'  🔍 Catégorie : {category}')

    # ── Étape 2 : Recherche RAG ──
    docs = tool_search(question, top_k=3)
    trace.append(f'[Étape 2] {len(docs)} documents récupérés')
    if verbose:
        print(f'  📚 {len(docs)} documents trouvés')
        for i, d in enumerate(docs):
            print(f'     [{i+1}] Score:{d["score"]:.3f} | {d["text"][:80]}...')

    # ── Étape 3 : Score de confiance ──
    confidence = tool_confidence(question, docs)
    trace.append(f'[Étape 3] Confiance : {confidence}')
    if verbose:
        print(f'  📊 Confiance RAG : {confidence}')

    # ── Étape 4 : Décision de l'agent ──
    if confidence > 0.3:
        strategy = 'RAG'
        context  = ' | '.join([d['text'][:150] for d in docs])
        trace.append('[Étape 4] Stratégie : RAG (confiance suffisante)')
    else:
        strategy = 'Direct'
        context  = f'3GPP {category} domain knowledge'
        trace.append('[Étape 4] Stratégie : Direct (confiance insuffisante)')
    if verbose:
        print(f'  🤖 Stratégie choisie : {strategy}')

    # ── Étape 5 : Génération ──
    answer = tool_generate(question, context)
    trace.append(f'[Étape 5] Réponse générée')
    elapsed = time.time() - t0

    return {
        'answer'    : answer,
        'category'  : category,
        'confidence': confidence,
        'strategy'  : strategy,
        'trace'     : trace,
        'time'      : round(elapsed, 2)
    }

print('✅ Agent RAG défini')

## 7. Test de l'Agent sur une Question

In [ ]:
# Test sur la première question
test_items = list(test_data.values())
test_q     = test_items[0]['question']

print(f'❓ Question : {test_q}')
print('\n🤖 Raisonnement de l\'Agent :')
print('='*60)

result = rag_agent(test_q, verbose=True)

print('='*60)
print(f'\n📋 Trace complète :')
for step in result['trace']:
    print(f'   {step}')
print(f'\n💬 Réponse finale : {result["answer"][:200]}')
print(f'⏱️  Temps total   : {result["time"]}s')

## 8. Évaluation de l'Agent sur 10 Questions

In [ ]:
results  = []
strategies = {'RAG': 0, 'Direct': 0}

for i, item in enumerate(test_items[:10]):
    question    = item['question']
    answer_idx  = item.get('answer', item.get('option 1', ''))
    answer_text = item.get(f'option {answer_idx}', str(answer_idx)) if isinstance(answer_idx, int) else str(answer_idx)

    print(f'\n🔍 Q{i+1} : {question[:60]}...')
    agent_result = rag_agent(question, verbose=False)
    strategies[agent_result['strategy']] += 1

    results.append({
        'id'        : i + 1,
        'question'  : question,
        'reference' : str(answer_text),
        'answer'    : agent_result['answer'],
        'category'  : agent_result['category'],
        'confidence': agent_result['confidence'],
        'strategy'  : agent_result['strategy'],
        'time'      : agent_result['time']
    })
    print(f'   ✓ Stratégie: {agent_result["strategy"]} | Confiance: {agent_result["confidence"]} | {agent_result["time"]}s')

df_results = pd.DataFrame(results)
df_results.to_csv(f'{SAVE_DIR}\\Phase6\\phase6_results.csv', index=False)
print(f'\n✅ {len(df_results)} résultats sauvegardés')
print(f'\n📊 Stratégies utilisées : {strategies}')

## 9. Évaluation ROUGE

In [ ]:
from evaluate import load as load_metric
rouge  = load_metric('rouge')
refs   = df_results['reference'].tolist()
preds  = df_results['answer'].tolist()
scores = rouge.compute(predictions=preds, references=refs)

print('📊 Scores ROUGE — Agent RAG :')
print(f'   ROUGE-1 : {round(scores["rouge1"], 4)}')
print(f'   ROUGE-2 : {round(scores["rouge2"], 4)}')
print(f'   ROUGE-L : {round(scores["rougeL"], 4)}')
print(f'   Temps moy : {df_results["time"].mean():.2f}s')

# Comparaison avec Phase 5
phase5_eval = pd.read_csv(f'{SAVE_DIR}\\Phase5\\phase5_evaluation.csv')
raft_rouge_l = phase5_eval[phase5_eval['Approche'] == 'RAFT']['ROUGE-L'].values[0]
agent_rouge_l = round(scores['rougeL'], 4)
delta = agent_rouge_l - raft_rouge_l
print(f'\n📊 Agent RAG vs RAFT :')
print(f'   RAFT      : {raft_rouge_l}')
print(f'   Agent RAG : {agent_rouge_l}')
print(f'   Différence : {delta:+.4f}')

df_eval = pd.DataFrame([{
    'Approche'      : 'Agent RAG',
    'ROUGE-1'       : round(scores['rouge1'], 4),
    'ROUGE-2'       : round(scores['rouge2'], 4),
    'ROUGE-L'       : agent_rouge_l,
    'Temps moy. (s)': round(df_results['time'].mean(), 2)
}])
df_eval.to_csv(f'{SAVE_DIR}\\Phase6\\phase6_evaluation.csv', index=False)
print('\n💾 Sauvegardé → Phase6/phase6_evaluation.csv')

## 10. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 6 — Agent RAG sur Dataset 3GPP', fontsize=15, fontweight='bold')

# --- Stratégies utilisées ---
ax1 = axes[0]
strat_counts = df_results['strategy'].value_counts()
ax1.pie(strat_counts.values, labels=strat_counts.index,
        colors=['#26A69A', '#FF7043'], autopct='%1.0f%%', startangle=90)
ax1.set_title('Stratégies utilisées par l\'Agent', fontweight='bold')

# --- Catégories détectées ---
ax2 = axes[1]
cat_counts = df_results['category'].value_counts()
ax2.barh(cat_counts.index, cat_counts.values, color='#7B1FA2', alpha=0.85)
ax2.set_title('Catégories 3GPP détectées', fontweight='bold')
ax2.set_xlabel('Nombre de questions')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}\\Phase6\\phase6_agent_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Graphique → Phase6/phase6_agent_analysis.png')

---
## ✅ Phase 6 Terminée — Agent RAG

**Ce qu'on a accompli :**
- ✅ Agent avec 4 outils (search, detect, generate, confidence)
- ✅ Raisonnement en 5 étapes
- ✅ Décision automatique de stratégie (RAG vs Direct)
- ✅ Traçabilité complète des actions

**Fichiers produits :**
- `Phase6/phase6_results.csv`
- `Phase6/phase6_evaluation.csv`
- `Phase6/phase6_agent_analysis.png`

**➡️ Prochaine étape : Phase 7 — Multi-Agents + RAFT**